# 🎯 Comprehensive Backtesting Analysis
## Quantitative Trading Strategy MVP Project

This notebook performs comprehensive backtesting with realistic constraints and advanced analytics.


In [1]:
# Setup and imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
from datetime import datetime, timedelta
from scipy import stats
import pickle
import sys
import asyncio
warnings.filterwarnings('ignore')

# Windows: ensure selector event loop policy for ZMQ compatibility
if sys.platform.startswith('win'):
    try:
        from asyncio import WindowsSelectorEventLoopPolicy
        asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())
    except Exception:
        pass

# Import custom modules
sys.path.append('../src')
from backtesting.backtest_engine import BacktestEngine
from backtesting.performance_metrics import PerformanceAnalyzer
from models.momentum_strategy import MomentumStrategy
from models.mean_reversion import MeanReversionStrategy
from data_collection.yfinance_collector import YFinanceCollector

print("🎯 Backtesting analysis setup complete!")


ModuleNotFoundError: No module named 'backtesting'

In [ ]:
# Load strategies from development phase
try:
    with open('../data/strategy_development_results.pkl', 'rb') as f:
        strategy_data = pickle.load(f)

    strategies = strategy_data['strategies']
    focus_data = strategy_data['focus_data']
    best_params = strategy_data.get('best_params')

    print("✅ Loaded strategy development results")
    print(f"   Available strategies: {list(strategies.keys())}")
    print(f"   Focus data shape: {focus_data.shape}")

except FileNotFoundError:
    print("⚠️ Strategy development results not found. Running basic setup...")

    # Create basic strategies for testing
    import yfinance as yf
    focus_data = yf.download('AAPL', start='2020-01-01', end='2024-01-01')

    momentum_strategy = MomentumStrategy(20, 50, 14, 0.1)
    mr_strategy = MeanReversionStrategy(20, 2.0, 0.5, 0.1)

    strategies = {
        'momentum_basic': {'strategy': momentum_strategy},
        'mean_reversion_basic': {'strategy': mr_strategy}
    }

    print("✅ Created basic strategies for testing")

# Initialize performance analyzer
performance_analyzer = PerformanceAnalyzer(risk_free_rate=0.02)

print(f"\n📊 Ready for comprehensive backtesting analysis")


In [ ]:
# Run backtests with transaction costs
backtest_engine = BacktestEngine(
    initial_capital=100000,
    commission_rate=0.001,
    slippage_rate=0.0005,
    borrowing_rate=0.03,
    min_commission=1.0
)

backtest_results = {}

for strategy_name, strategy_info in strategies.items():
    if 'strategy' not in strategy_info:
        continue
    
    print(f"\n📈 Backtesting {strategy_name}...")
    try:
        strategy = strategy_info['strategy']
        signals_df, basic_performance = strategy.backtest(focus_data)

        price_data = focus_data[['Close']].copy()
        signal_data = signals_df[['Signal']].copy()

        detailed_results = backtest_engine.run_backtest(price_data, signal_data)

        backtest_results[strategy_name] = {
            'signals': signals_df,
            'basic_performance': basic_performance,
            'detailed_results': detailed_results,
            'strategy_object': strategy
        }
        
        print(f"   ✅ {strategy_name}: {detailed_results['total_return']:.2%} return, {detailed_results['sharpe_ratio']:.2f} Sharpe")
    except Exception as e:
        print(f"   ❌ Error backtesting {strategy_name}: {str(e)}")

print(f"\n📊 Completed backtesting for {len(backtest_results)} strategies")


## 3. Walk-Forward Analysis (Out-of-Sample Testing)


In [ ]:
def walk_forward_analysis(data, strategy, training_months=12, testing_months=3, step_months=1):
    results = []
    data_monthly = data.resample('M').last()
    total_months = len(data_monthly)
    for i in range(training_months, total_months - testing_months, step_months):
        train_start = data_monthly.index[i - training_months]
        train_end = data_monthly.index[i - 1]
        test_start = data_monthly.index[i]
        test_end = data_monthly.index[min(i + testing_months - 1, total_months - 1)]
        train_data = data[train_start:train_end]
        test_data = data[test_start:test_end]
        if len(train_data) < 100 or len(test_data) < 20:
            continue
        try:
            optimized_strategy = strategy.optimize_parameters(train_data) if hasattr(strategy, 'optimize_parameters') else strategy
            test_signals, test_performance = optimized_strategy.backtest(test_data)
            benchmark_return = (test_data['Close'].iloc[-1] / test_data['Close'].iloc[0]) - 1
            results.append({
                'train_start': train_start,
                'train_end': train_end,
                'test_start': test_start,
                'test_end': test_end,
                'strategy_return': test_performance['total_return'],
                'benchmark_return': benchmark_return,
                'alpha': test_performance['total_return'] - benchmark_return,
                'sharpe_ratio': test_performance['sharpe_ratio'],
                'max_drawdown': test_performance['max_drawdown'],
                'total_trades': test_performance['total_trades'],
                'win_rate': test_performance['win_rate']
            })
        except Exception:
            continue
    return pd.DataFrame(results)

if backtest_results:
    print("🚶 Walk-Forward Analysis (Out-of-Sample Testing):")
    best_strategy_name = max(
        backtest_results.keys(),
        key=lambda k: backtest_results[k]['detailed_results']['sharpe_ratio']
    )
    best_strategy = backtest_results[best_strategy_name]['strategy_object']
    print(f"   Analyzing: {best_strategy_name}")
    print(f"   Training window: 12 months")
    print(f"   Testing window: 3 months")
    print(f"   Step size: 1 month")

    wf_results = walk_forward_analysis(focus_data, best_strategy, 12, 3, 1)
    if len(wf_results) > 0:
        print(f"   Completed {len(wf_results)} walk-forward iterations")
        wf_summary = {
            'avg_oos_return': wf_results['strategy_return'].mean(),
            'avg_benchmark_return': wf_results['benchmark_return'].mean(),
            'avg_alpha': wf_results['alpha'].mean(),
            'avg_sharpe': wf_results['sharpe_ratio'].mean(),
            'win_rate_oos': (wf_results['strategy_return'] > 0).mean(),
            'outperformance_rate': (wf_results['alpha'] > 0).mean(),
            'consistency_score': wf_results['sharpe_ratio'].std()
        }
        print("\n📊 Walk-Forward Results Summary:")
        for key, value in wf_summary.items():
            if 'rate' in key or 'score' in key:
                print(f"   {key.replace('_', ' ').title()}: {value:.1%}")
            else:
                print(f"   {key.replace('_', ' ').title()}: {value:.4f}")

        fig = make_subplots(rows=2, cols=2,
            subplot_titles=('Out-of-Sample Returns', 'Alpha Over Time', 'Sharpe Ratio Stability', 'Cumulative Alpha'),
            vertical_spacing=0.1)
        fig.add_trace(go.Scatter(x=wf_results['test_start'], y=wf_results['strategy_return'], name='Strategy Return', line=dict(color='blue')), row=1, col=1)
        fig.add_trace(go.Scatter(x=wf_results['test_start'], y=wf_results['benchmark_return'], name='Benchmark Return', line=dict(color='gray')), row=1, col=1)
        fig.add_trace(go.Scatter(x=wf_results['test_start'], y=wf_results['alpha'], name='Alpha', line=dict(color='green'), fill='tonexty'), row=1, col=2)
        fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)
        fig.add_trace(go.Scatter(x=wf_results['test_start'], y=wf_results['sharpe_ratio'], name='Sharpe Ratio', line=dict(color='purple')), row=2, col=1)
        cumulative_alpha = wf_results['alpha'].cumsum()
        fig.add_trace(go.Scatter(x=wf_results['test_start'], y=cumulative_alpha, name='Cumulative Alpha', line=dict(color='orange')), row=2, col=2)
        fig.update_layout(title='Walk-Forward Analysis Results', height=600, showlegend=True)
        fig.show()
        alpha_tstat, alpha_pvalue = stats.ttest_1samp(wf_results['alpha'], 0)
        print(f"\n🧪 Statistical Significance Test:")
        print(f"   Alpha t-statistic: {alpha_tstat:.3f}")
        print(f"   P-value: {alpha_pvalue:.4f}")
        print(f"   Statistically significant (p<0.05): {'Yes' if alpha_pvalue < 0.05 else 'No'}")
    else:
        print("   ⚠️ Insufficient data for walk-forward analysis")
else:
    print("⚠️ No backtest results available for walk-forward analysis")


## 4. Monte Carlo Validation and Bootstrap Analysis


In [ ]:
def monte_carlo_strategy_validation(strategy_returns, n_simulations=10000):
    clean_returns = strategy_returns.dropna()
    n_periods = len(clean_returns)
    observed_sharpe = clean_returns.mean() / clean_returns.std() * np.sqrt(252)
    observed_total_return = (1 + clean_returns).prod() - 1
    random_sharpes, random_returns = [], []
    strategy_vol = clean_returns.std()
    for _ in range(n_simulations):
        random_strategy_returns = np.random.normal(0, strategy_vol, n_periods)
        random_sharpes.append(random_strategy_returns.mean() / random_strategy_returns.std() * np.sqrt(252))
        random_returns.append((1 + random_strategy_returns).prod() - 1)
    sharpe_percentile = (np.array(random_sharpes) < observed_sharpe).mean() * 100
    return_percentile = (np.array(random_returns) < observed_total_return).mean() * 100
    return {
        'observed_sharpe': observed_sharpe,
        'observed_return': observed_total_return,
        'sharpe_percentile': sharpe_percentile,
        'return_percentile': return_percentile,
        'random_sharpes': random_sharpes,
        'random_returns': random_returns,
        'is_significant_sharpe': sharpe_percentile > 95,
        'is_significant_return': return_percentile > 95
    }

def bootstrap_confidence_intervals(returns, n_bootstrap=1000, confidence_level=0.95):
    bootstrap_sharpes = []
    bootstrap_returns = []
    clean_returns = returns.dropna()
    n_periods = len(clean_returns)
    for _ in range(n_bootstrap):
        sample = np.random.choice(clean_returns, size=n_periods, replace=True)
        bootstrap_sharpes.append(sample.mean() / sample.std() * np.sqrt(252))
        bootstrap_returns.append((1 + sample).prod() - 1)
    alpha = 1 - confidence_level
    return {
        'sharpe_ci': np.percentile(bootstrap_sharpes, [alpha/2*100, (1-alpha/2)*100]),
        'return_ci': np.percentile(bootstrap_returns, [alpha/2*100, (1-alpha/2)*100]),
        'bootstrap_sharpes': bootstrap_sharpes,
        'bootstrap_returns': bootstrap_returns
    }

print("🎲 Monte Carlo Validation and Bootstrap Analysis:")
mc_results = {}
bootstrap_results = {}
for strategy_name, results in backtest_results.items():
    print(f"\n📊 Analyzing {strategy_name}:")
    portfolio_history = results['detailed_results'].get('portfolio_history')
    if portfolio_history is not None and 'Returns' in portfolio_history.columns:
        strategy_returns = portfolio_history['Returns'].dropna()
        if len(strategy_returns) > 50:
            mc_result = monte_carlo_strategy_validation(strategy_returns)
            mc_results[strategy_name] = mc_result
            print(f"   Observed Sharpe: {mc_result['observed_sharpe']:.3f}")
            print(f"   Sharpe Percentile: {mc_result['sharpe_percentile']:.1f}%")
            print(f"   Statistically Significant: {'Yes' if mc_result['is_significant_sharpe'] else 'No'}")
            bootstrap_result = bootstrap_confidence_intervals(strategy_returns)
            bootstrap_results[strategy_name] = bootstrap_result
            print(f"   Sharpe 95% CI: [{bootstrap_result['sharpe_ci'][0]:.3f}, {bootstrap_result['sharpe_ci'][1]:.3f}]")
            print(f"   Return 95% CI: [{bootstrap_result['return_ci'][0]:.1%}, {bootstrap_result['return_ci'][1]:.1%}]")
        else:
            print("   ⚠️ Insufficient data for analysis")
    else:
        print("   ⚠️ No returns data available")

if mc_results:
    example_strategy = list(mc_results.keys())[0]
    mc_data = mc_results[example_strategy]
    bootstrap_data = bootstrap_results[example_strategy]
    fig = make_subplots(rows=2, cols=2,
        subplot_titles=('Monte Carlo Sharpe Distribution', 'Monte Carlo Return Distribution', 'Bootstrap Sharpe Distribution', 'Bootstrap Return Distribution'))
    fig.add_trace(go.Histogram(x=mc_data['random_sharpes'], name='Random Sharpes', opacity=0.7, nbinsx=50), row=1, col=1)
    fig.add_vline(x=mc_data['observed_sharpe'], line_dash='dash', line_color='red', annotation_text=f"Observed: {mc_data['observed_sharpe']:.2f}", row=1, col=1)
    fig.add_trace(go.Histogram(x=mc_data['random_returns'], name='Random Returns', opacity=0.7, nbinsx=50), row=1, col=2)
    fig.add_vline(x=mc_data['observed_return'], line_dash='dash', line_color='red', annotation_text=f"Observed: {mc_data['observed_return']:.1%}", row=1, col=2)
    fig.add_trace(go.Histogram(x=bootstrap_data['bootstrap_sharpes'], name='Bootstrap Sharpes', opacity=0.7, nbinsx=50), row=2, col=1)
    fig.add_vline(x=bootstrap_data['sharpe_ci'][0], line_dash='dot', line_color='green', row=2, col=1)
    fig.add_vline(x=bootstrap_data['sharpe_ci'][1], line_dash='dot', line_color='green', row=2, col=1)
    fig.add_trace(go.Histogram(x=bootstrap_data['bootstrap_returns'], name='Bootstrap Returns', opacity=0.7, nbinsx=50), row=2, col=2)
    fig.add_vline(x=bootstrap_data['return_ci'][0], line_dash='dot', line_color='green', row=2, col=2)
    fig.add_vline(x=bootstrap_data['return_ci'][1], line_dash='dot', line_color='green', row=2, col=2)
    fig.update_layout(title=f'Statistical Validation: {example_strategy}', height=600, showlegend=False)
    fig.show()
    validation_summary = []
    for strategy_name in mc_results.keys():
        mc_data = mc_results[strategy_name]
        bootstrap_data = bootstrap_results[strategy_name]
        validation_summary.append({
            'Strategy': strategy_name,
            'Observed_Sharpe': mc_data['observed_sharpe'],
            'Sharpe_Percentile': mc_data['sharpe_percentile'],
            'Statistically_Significant': mc_data['is_significant_sharpe'],
            'Sharpe_CI_Lower': bootstrap_data['sharpe_ci'][0],
            'Sharpe_CI_Upper': bootstrap_data['sharpe_ci'][1]
        })
    validation_df = pd.DataFrame(validation_summary)
    print("\n📋 Statistical Validation Summary:")
    display(validation_df.round(3))
else:
    print("\n⚠️ No suitable data for Monte Carlo validation")


## 5. Performance Attribution Analysis


In [ ]:
def performance_attribution_analysis(signals_df, market_data):
    attribution = {}
    market_returns = market_data['Close'].pct_change().dropna()
    if 'Strategy_Returns' in signals_df.columns:
        strategy_returns = signals_df['Strategy_Returns'].dropna()
        aligned_data = pd.DataFrame({
            'Strategy': strategy_returns,
            'Market': market_returns,
            'Position': signals_df['Position'] if 'Position' in signals_df.columns else 0
        }).dropna()
        if len(aligned_data) > 50:
            position_changes = aligned_data['Position'].diff()
            market_timing_returns = position_changes.shift(1) * aligned_data['Market']
            market_timing_contrib = market_timing_returns.sum()
            average_beta = aligned_data['Position'].mean()
            beta_timing_returns = (aligned_data['Position'] - average_beta) * aligned_data['Market']
            beta_timing_contrib = beta_timing_returns.sum()
            selection_returns = aligned_data['Strategy'] - aligned_data['Position'] * aligned_data['Market']
            selection_contrib = selection_returns.sum()
            total_strategy_return = aligned_data['Strategy'].sum()
            total_market_return = aligned_data['Market'].sum()
            attribution = {
                'total_strategy_return': total_strategy_return,
                'total_market_return': total_market_return,
                'excess_return': total_strategy_return - total_market_return,
                'market_timing_contrib': market_timing_contrib,
                'beta_timing_contrib': beta_timing_contrib,
                'security_selection_contrib': selection_contrib,
                'average_market_exposure': average_beta,
                'tracking_error': (aligned_data['Strategy'] - aligned_data['Market']).std() * np.sqrt(252),
                'information_ratio': (aligned_data['Strategy'] - aligned_data['Market']).mean() / (aligned_data['Strategy'] - aligned_data['Market']).std() * np.sqrt(252)
            }
            if attribution['excess_return'] != 0:
                attribution['market_timing_pct'] = market_timing_contrib / attribution['excess_return']
                attribution['beta_timing_pct'] = beta_timing_contrib / attribution['excess_return']
                attribution['selection_pct'] = selection_contrib / attribution['excess_return']
            else:
                attribution['market_timing_pct'] = 0
                attribution['beta_timing_pct'] = 0
                attribution['selection_pct'] = 0
    return attribution

print("🔍 Performance Attribution Analysis:")
attribution_results = {}
for strategy_name, results in backtest_results.items():
    print(f"\n📊 Attributing {strategy_name} performance:")
    signals_df = results['signals']
    attribution = performance_attribution_analysis(signals_df, focus_data)
    if attribution:
        attribution_results[strategy_name] = attribution
        print(f"   Total Strategy Return: {attribution['total_strategy_return']:.2%}")
        print(f"   Market Return: {attribution['total_market_return']:.2%}")
        print(f"   Excess Return: {attribution['excess_return']:.2%}")
        print(f"\n   Attribution Breakdown:")
        print(f"     Market Timing: {attribution['market_timing_contrib']:.2%} ({attribution['market_timing_pct']:.1%})")
        print(f"     Beta Timing: {attribution['beta_timing_contrib']:.2%} ({attribution['beta_timing_pct']:.1%})")
        print(f"     Security Selection: {attribution['security_selection_contrib']:.2%} ({attribution['selection_pct']:.1%})")
        print(f"\n   Information Ratio: {attribution['information_ratio']:.3f}")
        print(f"   Tracking Error: {attribution['tracking_error']:.2%}")
        print(f"   Average Market Exposure: {attribution['average_market_exposure']:.1%}")
    else:
        print("   ⚠️ Unable to perform attribution analysis")

if attribution_results:
    attribution_data = []
    for strategy_name, attr in attribution_results.items():
        attribution_data.extend([
            {'Strategy': strategy_name, 'Component': 'Market Timing', 'Contribution': attr['market_timing_contrib']},
            {'Strategy': strategy_name, 'Component': 'Beta Timing', 'Contribution': attr['beta_timing_contrib']},
            {'Strategy': strategy_name, 'Component': 'Security Selection', 'Contribution': attr['security_selection_contrib']}
        ])
    attribution_df = pd.DataFrame(attribution_data)
    fig = px.bar(attribution_df, x='Strategy', y='Contribution', color='Component', title='Performance Attribution Breakdown', labels={'Contribution': 'Contribution to Excess Return'}, barmode='relative')
    fig.show()
    risk_return_attr = []
    for strategy_name, attr in attribution_results.items():
        risk_return_attr.append({
            'Strategy': strategy_name,
            'Excess_Return': attr['excess_return'],
            'Tracking_Error': attr['tracking_error'],
            'Information_Ratio': attr['information_ratio'],
            'Market_Exposure': attr['average_market_exposure']
        })
    risk_return_df = pd.DataFrame(risk_return_attr)
    fig2 = px.scatter(risk_return_df, x='Tracking_Error', y='Excess_Return', size='Market_Exposure', color='Information_Ratio', text='Strategy', title='Risk-Return Attribution Analysis', labels={'Tracking_Error': 'Tracking Error (Annual)', 'Excess_Return': 'Excess Return', 'Information_Ratio': 'Information Ratio'})
    fig2.update_traces(textposition="top center")
    fig2.show()
    print("\n📋 Performance Attribution Summary:")
    display(risk_return_df.round(4))


## 6. Multi-Asset Portfolio Backtesting


In [ ]:
print("🌐 Multi-Asset Portfolio Backtesting:")
try:
    collector = YFinanceCollector()
    portfolio_tickers = ['AAPL', 'GOOGL', 'MSFT', 'SPY']
    print(f"   Downloading data for portfolio assets: {portfolio_tickers}")
    portfolio_data = collector.get_multiple_stocks(portfolio_tickers, '2020-01-01', '2024-01-01')
    if len(portfolio_data) >= 2:
        print(f"   Successfully loaded data for {len(portfolio_data)} assets")
        returns_matrix = pd.DataFrame()
        price_matrix = pd.DataFrame()
        for ticker, data in portfolio_data.items():
            returns_matrix[ticker] = data['Close'].pct_change()
            price_matrix[ticker] = data['Close']
        returns_matrix = returns_matrix.dropna()
        price_matrix = price_matrix.dropna()
        print(f"   Portfolio data shape: {returns_matrix.shape}")
        if backtest_results:
            best_strategy_name = max(backtest_results.keys(), key=lambda k: backtest_results[k]['detailed_results']['sharpe_ratio'])
            best_strategy = backtest_results[best_strategy_name]['strategy_object']
            print(f"   Applying {best_strategy_name} to portfolio assets")
            portfolio_signals = {}
            portfolio_performance = {}
            for ticker in portfolio_tickers:
                if ticker in portfolio_data:
                    try:
                        asset_data = portfolio_data[ticker]
                        signals, performance = best_strategy.backtest(asset_data)
                        portfolio_signals[ticker] = signals
                        portfolio_performance[ticker] = performance
                        print(f"     {ticker}: {performance['total_return']:.2%} return, {performance['sharpe_ratio']:.2f} Sharpe")
                    except Exception as e:
                        print(f"     ❌ Error with {ticker}: {str(e)}")
            if len(portfolio_signals) >= 2:
                equal_weights = {ticker: 1/len(portfolio_signals) for ticker in portfolio_signals.keys()}
                portfolio_strategy_returns = pd.DataFrame()
                for ticker, signals in portfolio_signals.items():
                    if 'Strategy_Returns' in signals.columns:
                        portfolio_strategy_returns[ticker] = signals['Strategy_Returns']
                weighted_returns = portfolio_strategy_returns * pd.Series(equal_weights)
                portfolio_total_returns = weighted_returns.sum(axis=1)
                portfolio_cumulative = (1 + portfolio_total_returns).cumprod()
                portfolio_metrics = {
                    'total_return': portfolio_cumulative.iloc[-1] - 1,
                    'annualized_return': portfolio_total_returns.mean() * 252,
                    'volatility': portfolio_total_returns.std() * np.sqrt(252),
                    'sharpe_ratio': (portfolio_total_returns.mean() / portfolio_total_returns.std()) * np.sqrt(252),
                    'max_drawdown': ((portfolio_cumulative / portfolio_cumulative.expanding().max()) - 1).min()
                }
                print(f"\n🎯 Portfolio Performance Summary:")
                print(f"   Total Return: {portfolio_metrics['total_return']:.2%}")
                print(f"   Annualized Return: {portfolio_metrics['annualized_return']:.2%}")
                print(f"   Volatility: {portfolio_metrics['volatility']:.2%}")
                print(f"   Sharpe Ratio: {portfolio_metrics['sharpe_ratio']:.3f}")
                print(f"   Max Drawdown: {portfolio_metrics['max_drawdown']:.2%}")
                individual_metrics = []
                for ticker, perf in portfolio_performance.items():
                    individual_metrics.append({'Asset': ticker, 'Return': perf['total_return'], 'Sharpe': perf['sharpe_ratio'], 'Max_DD': perf['max_drawdown']})
                individual_metrics.append({'Asset': 'PORTFOLIO', 'Return': portfolio_metrics['total_return'], 'Sharpe': portfolio_metrics['sharpe_ratio'], 'Max_DD': portfolio_metrics['max_drawdown']})
                portfolio_comparison_df = pd.DataFrame(individual_metrics)
                fig = make_subplots(rows=1, cols=3, subplot_titles=('Total Returns', 'Sharpe Ratios', 'Max Drawdowns'))
                colors = ['lightblue'] * (len(portfolio_comparison_df) - 1) + ['red']
                fig.add_trace(go.Bar(x=portfolio_comparison_df['Asset'], y=portfolio_comparison_df['Return'], name='Returns', marker_color=colors), row=1, col=1)
                fig.add_trace(go.Bar(x=portfolio_comparison_df['Asset'], y=portfolio_comparison_df['Sharpe'], name='Sharpe', marker_color=colors, showlegend=False), row=1, col=2)
                fig.add_trace(go.Bar(x=portfolio_comparison_df['Asset'], y=portfolio_comparison_df['Max_DD'], name='Max DD', marker_color=colors, showlegend=False), row=1, col=3)
                fig.update_layout(title='Individual Assets vs Portfolio Performance', height=400)
                fig.show()
                print("\n📊 Individual vs Portfolio Comparison:")
                display(portfolio_comparison_df.round(4))
                avg_individual_return = portfolio_comparison_df[portfolio_comparison_df['Asset'] != 'PORTFOLIO']['Return'].mean()
                avg_individual_sharpe = portfolio_comparison_df[portfolio_comparison_df['Asset'] != 'PORTFOLIO']['Sharpe'].mean()
                diversification_benefit = {
                    'return_improvement': portfolio_metrics['total_return'] - avg_individual_return,
                    'sharpe_improvement': portfolio_metrics['sharpe_ratio'] - avg_individual_sharpe,
                    'risk_reduction': portfolio_metrics['volatility'] - returns_matrix[list(portfolio_signals.keys())].std().mean() * np.sqrt(252)
                }
                print(f"\n🌟 Diversification Benefits:")
                print(f"   Return improvement: {diversification_benefit['return_improvement']:.2%}")
                print(f"   Sharpe improvement: {diversification_benefit['sharpe_improvement']:.3f}")
                print(f"   Risk reduction: {diversification_benefit['risk_reduction']:.2%}")
            else:
                print("   ⚠️ Insufficient signals for portfolio construction")
        else:
            print("   ⚠️ No strategy available for portfolio application")
    else:
        print("   ⚠️ Insufficient assets downloaded for portfolio analysis")
except Exception as e:
    print(f"   ❌ Error in multi-asset analysis: {str(e)}")
